In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yasa
import psutil
from psutil._common import bytes2human

from pathlib import Path
from scipy.io import loadmat
from mne.filter import resample
from itertools import chain
from tqdm import tqdm
from sys import getsizeof

from functions import get_sequences
from functions import phasic_rem_v3
from functions import create_name_cbd, create_name_rgs, create_name_os
from runtime_logger import logger_setup

CBD_DIR = "/home/nero/datasets/CBD/"
RGS_DIR = "/home/nero/datasets/RGS14/"
OS_DIR = "/home/nero/datasets/OSbasic/"

CBD_OVERVIEW_PATH = "overview.csv"
overview_df = pd.read_csv(CBD_OVERVIEW_PATH)

fs_cbd = 2500
fs_os = 2500
fs_rgs = 1000

targetFs = 500
n_down_cbd = fs_cbd/targetFs
n_down_rgs = fs_rgs/targetFs
n_down_os = fs_os/targetFs

phasic_tonic_idx = {}

phasic_tonic_freq = {
    'rat_id': [],
    'study_day': [],
    'condition': [],
    'treatment': [],
    'trial_num': [],
    'state': [],
    'count': [],
    'count_norm': [],
    'duration': []
    }

phasic_tonic_dur = {
    'rat_id': [],
    'study_day': [],
    'condition': [],
    'treatment': [],
    'trial_num': [],
    'state': [],
    'duration': [],
    }

logger = logger_setup()

# CBD Dataset

In [2]:
pattern = r"[\w-]+posttrial[\w-]+"
mapped = {}

for root, dirs, fils in os.walk(CBD_DIR):
    for dir in dirs:
        # Check if the directory is a post trial directory
        if re.match(pattern, dir, flags=re.IGNORECASE):
            dir = Path(os.path.join(root, dir))
            HPC_file = next(dir.glob("*HPC*continuous*"))
            states = next(dir.glob('*-states*'))
            mapped[str(states)] = str(HPC_file)

print("Number of trials: ", len(mapped))

Number of trials:  170


In [5]:
with tqdm(mapped.keys()) as t:
    for state in t:
        hpc = mapped[state]

        title = create_name_cbd(hpc, overview_df)
        t.set_postfix_str(title) # Set the title for the progress bar

        logger.debug("Loading: {0}".format(title))
        logger.debug("fname: {0}".format(state))
        
        metadata = {}
        metaname_list  = title.split('_')
        metadata["rat_id"]    = int(metaname_list[0][3:])
        metadata["study_day"] = int(metaname_list[1][2:])
        metadata["condition"] = metaname_list[2]
        metadata["treatment"] = int(metaname_list[3])
        metadata["trial_num"] = int(metaname_list[4][-1])

        # Load the LFP data
        lfpHPC = loadmat(hpc)['HPC']
        lfpHPC = lfpHPC.flatten()

        # Load the states
        hypno = loadmat(state)['states']
        hypno = hypno.flatten()

        logger.debug("LFP shape: {0}, size: {1}MB)".format(str(lfpHPC.shape), getsizeof(lfpHPC)//1024**2))
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))

        # Skip if no REM epoch is detected
        if(not (np.any(hypno == 5))):
            logger.debug("No REM detected. Skipping.")
            continue

        #%% downsample the data
        logger.debug("Downsampling")
        data_resample = resample(lfpHPC, down=n_down_cbd, method='fft', verbose="INFO")
        logger.debug("Finished downsampling.")
        
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
        
        del lfpHPC # Save memory

        # Remove artifacts
        logger.debug("Artifact removal.")
        art_std, _ = yasa.art_detect(data_resample, targetFs , window=1, method='std', threshold=4, verbose='info')
        art_up = yasa.hypno_upsample_to_data(art_std, 1, data_resample, targetFs)
        data_resample[art_up] = 0

        del art_up # Save memory
        
        seq = get_sequences(np.where(hypno==5)[0])

        logger.debug("Number of REM epochs: {0}".format(len(seq)))
        logger.debug("Resampled data shape: {0}".format(data_resample.shape))
        logger.debug("Sleep states shape: {0}".format(hypno.shape))
    
        phasicREM = phasic_rem_v3(data_resample-np.mean(data_resample), hypno, targetFs, min_dur=2.5, vm = 1000, thr_dur = 900, nfilt=11)
    
        ######### Save phasic/tonic REM indices #####################
        rem_idx = {}

        # Create a dict of start and end indices for each rem epoch
        for s in seq:
            rem_idx[s[0]] = s[-1]

        phasic = []
        tonic = []

        for rem_start in phasicREM:
            rem_end = rem_idx[rem_start]
            phasic += phasicREM[rem_start]

            tonic_idx = [rem_start*targetFs] + list(chain.from_iterable(phasicREM[rem_start])) + [rem_end*targetFs]
            tonic += list(zip(tonic_idx[::2], tonic_idx[1::2]))
            logger.debug("REM epoch: ({0}, {1}) ".format(rem_start*targetFs, rem_end*targetFs))

        idx = {}
        idx["phasic"] = phasic
        idx["tonic"] = tonic
        phasic_tonic_idx[title] = idx
        #############################################################
    
        logger.debug("Phasic REM true indices: {0}".format(phasic))
        logger.debug("Tonic REM true indices: {0}".format(tonic))
    
        ###### Save phasic/tonic frequency and duration #############
        for state in ["phasic", "tonic"]:
            # Add metadata
            for condition in metadata.keys():
                phasic_tonic_freq[condition].append(metadata[condition])

            phasic_tonic_freq["state"].append(state)

            dur = [(ep[1] - ep[0])/targetFs for ep in idx[state]] # list of duration in seconds

            for d in dur:
                # Add the durations individually
                for condition in metadata.keys():
                    phasic_tonic_dur[condition].append(metadata[condition])
                phasic_tonic_dur["state"].append(state)
                phasic_tonic_dur["duration"].append(d)
            
            # Add total duration per REM epoch and count 
            phasic_tonic_freq["duration"].append(np.sum(dur))
            phasic_tonic_freq["count"].append(len(dur))

            # Count normalized by duration
            if (metadata["trial_num"] == 5):
                phasic_tonic_freq["count_norm"].append(len(dur)/180)
            else:
                phasic_tonic_freq["count_norm"].append(len(dur)/45)

            if(state == "phasic"):
                logger.debug("Phasic REM duration: {0}".format(dur))
            else:
                logger.debug("Tonic REM duration: {0}".format(dur))

            logger.debug("Total: {0} s".format(phasic_tonic_freq["duration"][-1]))
            logger.debug("Count: {0}".format(phasic_tonic_freq["count"][-1]))
            logger.debug("Count normalized: {0}".format(phasic_tonic_freq["count_norm"][-1]))
            #############################################################        

  0%|          | 0/170 [00:00<?, ?it/s, Rat5_SD8_HC_0_posttrial3]

  0%|          | 0/170 [00:01<?, ?it/s, Rat5_SD8_HC_0_posttrial3]


# RGS14 Dataset

In [4]:
pattern1 = r"[\w-]+post[\w-]+trial[\w-]+"
mapped = {}

for root, dirs, fils in os.walk(RGS_DIR):
    for dir in dirs:
        # Check if the directory is a post trial directory
        if re.match(pattern1, dir, flags=re.IGNORECASE):
            dir = Path(os.path.join(root, dir))
            HPC_file = next(dir.glob("*HPC*continuous*"))
            states = next(dir.glob('*-states*'))
            mapped[str(states)] = str(HPC_file)

print("Number of trials: ", len(mapped))

Number of trials:  166


In [5]:
with tqdm(mapped.keys()) as t:
    for state in t:
        hpc = mapped[state]

        title = create_name_rgs(hpc)
        t.set_postfix_str(title) # Set the title for the progress bar

        logger.debug("Loading: {0}".format(title))
        logger.debug("fname: {0}".format(state))
        
        metadata = {}
        metaname_list  = title.split('_')
        metadata["rat_id"]    = int(metaname_list[0][3:])
        metadata["study_day"] = int(metaname_list[1][2:])
        metadata["condition"] = metaname_list[2]
        metadata["treatment"] = int(metaname_list[3])
        metadata["trial_num"] = int(metaname_list[4][-1])

        # Load the LFP data
        lfpHPC = loadmat(hpc)['HPC']
        lfpHPC = lfpHPC.flatten()

        # Load the states
        hypno = loadmat(state)['states']
        hypno = hypno.flatten()

        logger.debug("LFP shape: {0}, size: {1}MB)".format(str(lfpHPC.shape), getsizeof(lfpHPC)//1024**2))
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))

        # Skip if no REM epoch is detected
        if(not (np.any(hypno == 5))):
            logger.debug("No REM detected. Skipping.")
            continue

        #%% downsample the data
        logger.debug("Downsampling")
        data_resample = resample(lfpHPC, down=n_down_rgs, method='fft', verbose="INFO")
        logger.debug("Finished downsampling.")
        
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
        
        del lfpHPC # Save memory

        # Remove artifacts
        logger.debug("Artifact removal.")
        art_std, _ = yasa.art_detect(data_resample, targetFs , window=1, method='std', threshold=4, verbose='info')
        art_up = yasa.hypno_upsample_to_data(art_std, 1, data_resample, targetFs)
        data_resample[art_up] = 0

        del art_up # Save memory

        seq = get_sequences(np.where(hypno==5)[0])

        logger.debug("Number of REM epochs: {0}".format(len(seq)))
        logger.debug("Resampled data shape: {0}".format(data_resample.shape))
        logger.debug("Sleep states shape: {0}".format(hypno.shape))
    
        phasicREM = phasic_rem_v3(data_resample-np.mean(data_resample), hypno, targetFs, min_dur=2.5, vm = 1000, thr_dur = 900, nfilt=11)
    
        ######### Save phasic/tonic REM indices #####################
        rem_idx = {}

        # Create a dict of start and end indices for each rem epoch
        for s in seq:
            rem_idx[s[0]] = s[-1]

        phasic = []
        tonic = []

        for rem_start in phasicREM:
            rem_end = rem_idx[rem_start]
            phasic += phasicREM[rem_start]

            tonic_idx = [rem_start*targetFs] + list(chain.from_iterable(phasicREM[rem_start])) + [rem_end*targetFs]
            tonic += list(zip(tonic_idx[::2], tonic_idx[1::2]))
            logger.debug("REM epoch: ({0}, {1}) ".format(rem_start*targetFs, rem_end*targetFs))

        idx = {}
        idx["phasic"] = phasic
        idx["tonic"] = tonic
        phasic_tonic_idx[title] = idx
        #############################################################
    
        logger.debug("Phasic REM true indices: {0}".format(phasic))
        logger.debug("Tonic REM true indices: {0}".format(tonic))
    
        ###### Save phasic/tonic frequency and duration #############
        for state in ["phasic", "tonic"]:
            # Add metadata
            for condition in metadata.keys():
                phasic_tonic_freq[condition].append(metadata[condition])

            phasic_tonic_freq["state"].append(state)

            dur = [(ep[1] - ep[0])/targetFs for ep in idx[state]] # list of duration in seconds

            for d in dur:
                # Add the durations individually
                for condition in metadata.keys():
                    phasic_tonic_dur[condition].append(metadata[condition])
                phasic_tonic_dur["state"].append(state)
                phasic_tonic_dur["duration"].append(d)

            # Add total duration per REM epoch and count 
            phasic_tonic_freq["duration"].append(np.sum(dur))
            phasic_tonic_freq["count"].append(len(dur))

            # Count normalized by duration
            if (metadata["trial_num"] == 5):
                phasic_tonic_freq["count_norm"].append(len(dur)/180)
            else:
                phasic_tonic_freq["count_norm"].append(len(dur)/45)

            if(state == "phasic"):
                logger.debug("Phasic REM duration: {0}".format(dur))
            else:
                logger.debug("Tonic REM duration: {0}".format(dur))

            logger.debug("Total: {0} s".format(phasic_tonic_freq["duration"][-1]))
            logger.debug("Count: {0}".format(phasic_tonic_freq["count"][-1]))
            logger.debug("Count normalized: {0}".format(phasic_tonic_freq["count_norm"][-1]))
            #############################################################
        

  0%|          | 0/166 [00:00<?, ?it/s, Rat2_SD3_CON_2_posttrial1]

  0%|          | 0/166 [00:00<?, ?it/s, Rat2_SD3_CON_2_posttrial1]


# OSbasic Dataset

In [6]:
pattern2 = r".*post_trial.*"
mapped = {}

for root, dirs, fils in os.walk(OS_DIR):
    for dir in dirs:
        dir = (os.path.join(root, dir))
        # Check if the directory is a post trial directory
        if re.match(pattern2, dir, flags=re.IGNORECASE):
            dir = Path(dir)
            HPC_file = next(dir.glob("*HPC*"))
            states = next(dir.glob('*states*'))
            mapped[str(states)] = str(HPC_file)

print("Number of trials: ", len(mapped))


Number of trials:  210


In [7]:
with tqdm(mapped.keys()) as t:
    for state in t:
        hpc = mapped[state]

        title = create_name_os(hpc)
        t.set_postfix_str(title) # Set the title for the progress bar

        logger.debug("Loading: {0}".format(title))
        logger.debug("fname: {0}".format(state))
        
        metadata = {}
        metaname_list  = title.split('_')
        metadata["rat_id"]    = int(metaname_list[0][3:])
        metadata["study_day"] = int(metaname_list[1][2:])
        metadata["condition"] = metaname_list[2]
        metadata["treatment"] = int(metaname_list[3])
        metadata["trial_num"] = int(metaname_list[4][-1])

        # Load the LFP data
        lfpHPC = loadmat(hpc)['HPC']
        lfpHPC = lfpHPC.flatten()

        # Load the states
        hypno = loadmat(state)['states']
        hypno = hypno.flatten()

        logger.debug("LFP shape: {0}, size: {1}MB)".format(str(lfpHPC.shape), getsizeof(lfpHPC)//1024**2))
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))

        # Skip if no REM epoch is detected
        if(not (np.any(hypno == 5))):
            logger.debug("No REM detected. Skipping.")
            continue

        #%% downsample the data
        logger.debug("Downsampling")
        data_resample = resample(lfpHPC, down=n_down_os, method='fft', verbose="INFO")
        logger.debug("Finished downsampling.")
        
        logger.debug("Memory usage: {0} ({1}%)".format(bytes2human(psutil.virtual_memory().available), str(psutil.virtual_memory().percent)))
        
        del lfpHPC # Save memory

        # Remove artifacts
        logger.debug("Artifact removal.")
        art_std, _ = yasa.art_detect(data_resample, targetFs , window=1, method='std', threshold=4, verbose='info')
        art_up = yasa.hypno_upsample_to_data(art_std, 1, data_resample, targetFs)
        data_resample[art_up] = 0

        del art_up # Save memory

        seq = get_sequences(np.where(hypno==5)[0])

        logger.debug("Number of REM epochs: {0}".format(len(seq)))
        logger.debug("Resampled data shape: {0}".format(data_resample.shape))
        logger.debug("Sleep states shape: {0}".format(hypno.shape))
    
        phasicREM = phasic_rem_v3(data_resample-np.mean(data_resample), hypno, targetFs, min_dur=2.5, vm = 1000, thr_dur = 900, nfilt=11)
    
        ######### Save phasic/tonic REM indices #####################
        rem_idx = {}

        # Create a dict of start and end indices for each rem epoch
        for s in seq:
            rem_idx[s[0]] = s[-1]

        phasic = []
        tonic = []

        for rem_start in phasicREM:
            rem_end = rem_idx[rem_start]
            phasic += phasicREM[rem_start]

            tonic_idx = [rem_start*targetFs] + list(chain.from_iterable(phasicREM[rem_start])) + [rem_end*targetFs]
            tonic += list(zip(tonic_idx[::2], tonic_idx[1::2]))
            logger.debug("REM epoch: ({0}, {1}) ".format(rem_start*targetFs, rem_end*targetFs))

        idx = {}
        idx["phasic"] = phasic
        idx["tonic"] = tonic
        phasic_tonic_idx[title] = idx
        #############################################################
    
        logger.debug("Phasic REM true indices: {0}".format(phasic))
        logger.debug("Tonic REM true indices: {0}".format(tonic))
    
        ###### Save phasic/tonic frequency and duration #############
        for state in ["phasic", "tonic"]:
            # Add metadata
            for condition in metadata.keys():
                phasic_tonic_freq[condition].append(metadata[condition])

            phasic_tonic_freq["state"].append(state)

            dur = [(ep[1] - ep[0])/targetFs for ep in idx[state]] # list of duration in seconds

            for d in dur:
                # Add the durations individually
                for condition in metadata.keys():
                    phasic_tonic_dur[condition].append(metadata[condition])
                phasic_tonic_dur["state"].append(state)
                phasic_tonic_dur["duration"].append(d)

            # Add total duration per REM epoch and count 
            phasic_tonic_freq["duration"].append(np.sum(dur))
            phasic_tonic_freq["count"].append(len(dur))

            # Count normalized by duration
            if (metadata["trial_num"] == 5):
                phasic_tonic_freq["count_norm"].append(len(dur)/180)
            else:
                phasic_tonic_freq["count_norm"].append(len(dur)/45)

            if(state == "phasic"):
                logger.debug("Phasic REM duration: {0}".format(dur))
            else:
                logger.debug("Tonic REM duration: {0}".format(dur))

            logger.debug("Total: {0} s".format(phasic_tonic_freq["duration"][-1]))
            logger.debug("Count: {0}".format(phasic_tonic_freq["count"][-1]))
            logger.debug("Count normalized: {0}".format(phasic_tonic_freq["count_norm"][-1]))
            #############################################################
        

  0%|          | 0/210 [00:00<?, ?it/s, Rat1_SD1_OD_4_posttrial5]

  0%|          | 0/210 [00:25<?, ?it/s, Rat1_SD1_OD_4_posttrial5]


# Save

In [8]:
np.save('phasic_tonic_idx', phasic_tonic_idx)
# For loading use
# data = np.load('data.npy', allow_pickle='TRUE').item()

ep_df = pd.DataFrame({key:pd.Series(value) for key, value in phasic_tonic_freq.items()})
ep_df.to_csv("states_freq_dur.csv", index=False)

dur_df = pd.DataFrame({key:pd.Series(value) for key, value in phasic_tonic_dur.items()})
dur_df.to_csv("states_durations.csv", index=False)

In [13]:
ep_df

,rat_id,study_day,condition,treatment,trial_num,state,count,count_norm,duration
0,5,8,HC,0,3,phasic,4,0.088889,10.270
1,5,8,HC,0,3,tonic,6,0.133333,190.730
2,2,3,CON,2,1,phasic,2,0.044444,2.470
3,2,3,CON,2,1,tonic,4,0.088889,109.530
4,1,1,OD,4,5,phasic,28,0.155556,54.548
5,1,1,OD,4,5,tonic,37,0.205556,1250.452


In [11]:
dur_df

,rat_id,study_day,condition,treatment,trial_num,state,duration
0,5,8,HC,0,3,phasic,1.784
1,5,8,HC,0,3,phasic,3.492
2,5,8,HC,0,3,phasic,2.626
3,5,8,HC,0,3,phasic,2.368
4,5,8,HC,0,3,tonic,48.982
...,...,...,...,...,...,...,...
76,1,1,OD,4,5,tonic,17.536
77,1,1,OD,4,5,tonic,17.310
78,1,1,OD,4,5,tonic,55.784
79,1,1,OD,4,5,tonic,65.288
